# Document Question Answering System using RAG

This project builds a simple Retrieval-Augmented Generation (RAG) system for answering questions from a custom PDF resume.

**Workflow:**  
Document Upload → Text Extraction → Chunking → Embeddings → FAISS Vector Search → Retrieval → Language Model Answer

In [52]:
!pip -q install pypdf sentence-transformers faiss-cpu pandas

Upload Custom Document - Upload a custom PDF or text file. In this project, a resume PDF is used as the private document source.

In [53]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for file_name in uploaded:
    print("-", file_name)

Saving SaanviBalki_Resume.pdf to SaanviBalki_Resume (1).pdf
Uploaded files:
- SaanviBalki_Resume (1).pdf


2. Document Ingestion and Text Extraction - The uploaded PDF is read page by page. Extracted text is stored with its source filename and page number so that retrieved answers can be traced back to the original document.

In [54]:
from pypdf import PdfReader

documents = []

for file_name in uploaded:
    if file_name.lower().endswith(".pdf"):
        reader = PdfReader(file_name)

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""

            if text.strip():
                documents.append({
                    "source": file_name,
                    "page": page_number,
                    "text": text
                })

    elif file_name.lower().endswith(".txt"):
        with open(file_name, "r", encoding="utf-8", errors="ignore") as file:
            text = file.read()

        documents.append({
            "source": file_name,
            "page": 1,
            "text": text
        })

print(f"Loaded {len(documents)} page(s)/text file(s).")

for doc in documents[:2]:
    print("\nSource:", doc["source"], "| Page:", doc["page"])
    print(doc["text"][:300])

Loaded 1 page(s)/text file(s).

Source: SaanviBalki_Resume (1).pdf | Page: 1
Saanvi Sunil Balki 
Pune, Maharashtra | +91 9372039940 | saanvibalki@gmail.com | www.linkedin.com/in/saanvi-balki | https://github.com/saanvib27 
EDUCATION  
Master of Technology in Data Science and Analytics                                                                                            


3. Text Chunking - Long document text is split into smaller overlapping chunks.

- Chunk size: 100 words  
- Chunk overlap: 20 words  

Chunking improves retrieval because the system searches focused pieces of text instead of the full document.

In [55]:
chunk_size = 100      # words in each chunk
chunk_overlap = 20   # repeated words between chunks

chunks = []

for doc in documents:
    words = doc["text"].split()

    for start in range(0, len(words), chunk_size - chunk_overlap):
        chunk_words = words[start:start + chunk_size]

        if len(chunk_words) < 30:
            continue

        chunks.append({
            "source": doc["source"],
            "page": doc["page"],
            "text": " ".join(chunk_words)
        })

print("Total chunks created:", len(chunks))
print("\nFirst chunk preview:\n", chunks[0]["text"][:500])

Total chunks created: 5

First chunk preview:
 Saanvi Sunil Balki Pune, Maharashtra | +91 9372039940 | saanvibalki@gmail.com | www.linkedin.com/in/saanvi-balki | https://github.com/saanvib27 EDUCATION Master of Technology in Data Science and Analytics Current MIT WPU, Pune (CGPA – 7.56) Bachelor of Technology in Computer Science Engineering 2020 - 2024 University of Mumbai (CGPA - 8.42) TECHNICAL SKILLS Programming Languages: Python, Java Frameworks & Tools: React.js, Node.js, Express.js, MySQL, OpenCV, GitHub, REST APIs, Socket.IO Technolog


4. Embeddings and FAISS Vector Database

Each text chunk is converted into a numerical vector using the `all-MiniLM-L6-v2` embedding model.

The vectors are stored in FAISS, a vector database that performs fast similarity search.

In [56]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(embedding_model_name)

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)
index.add(embeddings)

print("Embedding model:", embedding_model_name)
print("Embedding dimension:", embedding_dimension)
print("Vector database: FAISS IndexFlatIP")
print("Vectors stored:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Vector database: FAISS IndexFlatIP
Vectors stored: 5


5. Query Processing and Context Retrieval

When a user asks a question, the system converts the question into an embedding and compares it with stored document vectors.

The top 3 most similar chunks are retrieved as context for answer generation.

In [57]:
TOP_K = 3

def retrieve_chunks(question, top_k=TOP_K):
    question_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    actual_k = min(top_k, len(chunks))

    scores, indices = index.search(question_embedding, actual_k)

    results = []

    for score, chunk_index in zip(scores[0], indices[0]):
        if chunk_index == -1:
            continue

        results.append({
            "score": float(score),
            "source": chunks[chunk_index]["source"],
            "page": chunks[chunk_index]["page"],
            "text": chunks[chunk_index]["text"]
        })

    return results


test_question = "What are Saanvi Balki's technical skills?"

results = retrieve_chunks(test_question)

for number, result in enumerate(results, start=1):
    print(f"\nResult {number} | Similarity score: {result['score']:.3f}")
    print(f"Source: {result['source']} | Page: {result['page']}")
    print(result["text"][:400])


Result 1 | Similarity score: 0.464
Source: SaanviBalki_Resume (1).pdf | Page: 1
Saanvi Sunil Balki Pune, Maharashtra | +91 9372039940 | saanvibalki@gmail.com | www.linkedin.com/in/saanvi-balki | https://github.com/saanvib27 EDUCATION Master of Technology in Data Science and Analytics Current MIT WPU, Pune (CGPA – 7.56) Bachelor of Technology in Computer Science Engineering 2020 - 2024 University of Mumbai (CGPA - 8.42) TECHNICAL SKILLS Programming Languages: Python, Java Fram

Result 2 | Similarity score: 0.258
Source: SaanviBalki_Resume (1).pdf | Page: 1
| Data Science Intern June 2026 – August 2026 • Built Machine Learning pipelines using Python, Pandas, NumPy, and Scikit-learn, including EDA, preprocessing, feature engineering, and model evaluation. • Implemented a memory-augmented chatbot using RAG, LLMs, vector embeddings, and Agentic AI for context-aware conversational assistance. MIT WPU | Teaching Assistant Feb 2026 – May 2026 • Conducted P

Result 3 | Similarity score: 0.223


6. Language Model Setup - A free Hugging Face language model, `google/flan-t5-base`, is loaded for answer generation.
This model receives the retrieved document context and generates an answer based only on that context. No paid API is required.

In [58]:
!pip -q install -U transformers sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)

def generate_with_flan(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    output = llm_model.generate(
        **inputs,
        max_new_tokens=82,
        do_sample=False
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

print(f"Free language model is ready on: {device}")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Free language model is ready on: cpu


7. Grounded Answer Generation - The retrieved chunks and the user question are combined into a prompt for the language model.
The model is instructed to answer only using the retrieved resume content. This reduces hallucination and keeps answers grounded in the uploaded document.

In [59]:
validation_logs = []

def ask_rag(question):
    retrieved = retrieve_chunks(question)

    context = "\n\n".join(
        f"[Source: {item['source']} | Page: {item['page']}]\n{item['text']}"
        for item in retrieved
    )

    prompt = f"""
Use only the context below to answer the question.

Give only the direct answer in one short paragraph or bullet list.
Do not include unrelated resume sections.
Do not add information not present in the context.
If the answer is missing, say:
I cannot answer that from the provided document.

Context:
{context}

Question: {question}

Short answer:
"""

    answer = generate_with_flan(prompt)
    # Remove a trailing resume section heading, if the model starts copying one.
    for marker in [" EXPERIENCE", " EXPER", " PROJECT", " EDUCATION", " CERTIFICATION"]:
        position = answer.upper().find(marker)

        if position != -1:
           answer = answer[:position].strip()
           break

    validation_logs.append({
        "question": question,
        "answer": answer,
        "top_similarity_score": round(retrieved[0]["score"], 3),
        "sources": ", ".join(
            f"{item['source']} (page {item['page']})"
            for item in retrieved
        )
    })

    print("Question:", question)
    print("\nAnswer:\n", answer)

    print("\nRetrieved chunks:")
    for number, item in enumerate(retrieved, start=1):
        print(
            f"{number}. Score = {item['score']:.3f} | "
            f"{item['source']} | Page {item['page']}"
        )

    return answer

Test Questions - The following questions test whether the RAG system can retrieve relevant resume information and generate accurate answers.

In [60]:
ask_rag("What are Saanvi Balki's technical skills?")

Question: What are Saanvi Balki's technical skills?

Answer:
 Programming Languages: Python, Java Frameworks & Tools: React.js, Node.js, Express.js, MySQL, OpenCV, GitHub, REST APIs, Socket.IO Technologies: Machine Learning, RAG, LLMs, Model Deployment, Generative AI, ETL Pipelines, Deep Learning

Retrieved chunks:
1. Score = 0.464 | SaanviBalki_Resume (1).pdf | Page 1
2. Score = 0.258 | SaanviBalki_Resume (1).pdf | Page 1
3. Score = 0.223 | SaanviBalki_Resume (1).pdf | Page 1


'Programming Languages: Python, Java Frameworks & Tools: React.js, Node.js, Express.js, MySQL, OpenCV, GitHub, REST APIs, Socket.IO Technologies: Machine Learning, RAG, LLMs, Model Deployment, Generative AI, ETL Pipelines, Deep Learning'

In [61]:
ask_rag("What is Saanvi Balki's educational background?")

Question: What is Saanvi Balki's educational background?

Answer:
 Master of Technology in Data Science and Analytics.

Retrieved chunks:
1. Score = 0.335 | SaanviBalki_Resume (1).pdf | Page 1
2. Score = 0.142 | SaanviBalki_Resume (1).pdf | Page 1
3. Score = 0.103 | SaanviBalki_Resume (1).pdf | Page 1


'Master of Technology in Data Science and Analytics.'

In [62]:
ask_rag("What programming languages does Saanvi Balki know?")

Question: What programming languages does Saanvi Balki know?

Answer:
 Python, Java Frameworks & Tools: React.js, Node.js, Express.js, MySQL, OpenCV, GitHub, REST APIs, Socket.IO Technologies: Machine Learning, RAG, LLMs, Model Deployment, Generative AI, ETL Pipelines, Deep Learning

Retrieved chunks:
1. Score = 0.490 | SaanviBalki_Resume (1).pdf | Page 1
2. Score = 0.357 | SaanviBalki_Resume (1).pdf | Page 1
3. Score = 0.247 | SaanviBalki_Resume (1).pdf | Page 1


'Python, Java Frameworks & Tools: React.js, Node.js, Express.js, MySQL, OpenCV, GitHub, REST APIs, Socket.IO Technologies: Machine Learning, RAG, LLMs, Model Deployment, Generative AI, ETL Pipelines, Deep Learning'

Validation Logs

Validation logs record the tested questions, generated answers, similarity scores, and source document details.

These logs show that the system retrieves relevant context before generating responses.

In [63]:
import pandas as pd

log_table = pd.DataFrame(validation_logs)
display(log_table)

,question,answer,top_similarity_score,sources
0,What are Saanvi Balki's technical skills?,"Programming Languages: Python, Java Frameworks...",0.464,"SaanviBalki_Resume (1).pdf (page 1), SaanviBal..."
1,What is Saanvi Balki's educational background?,Master of Technology in Data Science and Analy...,0.335,"SaanviBalki_Resume (1).pdf (page 1), SaanviBal..."
2,What programming languages does Saanvi Balki k...,"Python, Java Frameworks & Tools: React.js, Nod...",0.490,"SaanviBalki_Resume (1).pdf (page 1), SaanviBal..."


System Metrics Report

This report summarizes the RAG configuration, including document count, chunking settings, embedding model, vector database, retrieval strategy, language model, and number of tested questions.

In [64]:
print("=" * 55)
print("SYSTEM METRICS REPORT")
print("=" * 55)

print("Input document:", list(uploaded.keys()))
print("Pages/text records processed:", len(documents))
print("Total chunks created:", len(chunks))
print("Chunk size:", chunk_size, "words")
print("Chunk overlap:", chunk_overlap, "words")
print("Embedding model:", embedding_model_name)
print("Embedding dimension:", embedding_dimension)
print("Vector store:", "FAISS IndexFlatIP")
print("Retrieval method:", "Dense vector similarity search")
print("Top-K retrieved chunks:", TOP_K)
print("Language model:", "google/flan-t5-base")
print("Questions tested:", len(validation_logs))

print("=" * 55)

SYSTEM METRICS REPORT
Input document: ['SaanviBalki_Resume (1).pdf']
Pages/text records processed: 1
Total chunks created: 5
Chunk size: 100 words
Chunk overlap: 20 words
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Vector store: FAISS IndexFlatIP
Retrieval method: Dense vector similarity search
Top-K retrieved chunks: 3
Language model: google/flan-t5-base
Questions tested: 3


Chunk-Size Optimization Experiment

This experiment compares the number of chunks created using different chunk sizes.

The selected configuration uses 100-word chunks with a 20-word overlap because it creates focused chunks for resume-based question answering.

In [65]:
def count_chunks(test_chunk_size, test_overlap=20):
    total_chunks = 0

    for doc in documents:
        words = doc["text"].split()

        for start in range(0, len(words), test_chunk_size - test_overlap):
            current_chunk = words[start:start + test_chunk_size]

            if len(current_chunk) >= 30:
                total_chunks += 1

    return total_chunks


print("CHUNK-SIZE EXPERIMENT")

for size in [80, 100, 150]:
    print(f"Chunk size {size} words -> {count_chunks(size)} chunks")

print("\nSelected configuration: 100 words with 20-word overlap.")
print("Reason: it creates enough focused resume chunks for accurate retrieval.")

CHUNK-SIZE EXPERIMENT
Chunk size 80 words -> 6 chunks
Chunk size 100 words -> 5 chunks
Chunk size 150 words -> 3 chunks

Selected configuration: 100 words with 20-word overlap.
Reason: it creates enough focused resume chunks for accurate retrieval.


## Conclusion

This project implements a simple Retrieval-Augmented Generation (RAG) system for answering questions from a custom PDF resume. The system extracts document text, splits it into chunks, converts chunks into embeddings, stores them in FAISS, retrieves relevant chunks for a user question, and generates a grounded response using a language model.

The chunk-size experiment showed that 100-word chunks with a 20-word overlap provide focused retrieval for this document.